# Interopérabilité avec C : Cython

Cette démonstration calcule la somme cumulée de la démonstration `ctypes`, cette fois en Cython : le code ressemble à du Python, avec des types, et Cython le traduit en C. Elle le compile de deux façons, dans le notebook avec `%%cython`, puis en module avec `cythonize -i`, et le chronomètre comme la démonstration `ctypes`. La première cellule charge l'extension `cython` d'IPython, qui ajoute la commande magique `%%cython`.

In [1]:
import numpy as np

%load_ext cython

## La version Python, pour référence

Une boucle Python, sur une liste d'un million de valeurs :

In [2]:
def cumsum_python(valeurs):
  b = []
  total = 0.0
  for x in valeurs:
    total += x
    b.append(total)
  return b


a = np.linspace(-1, 1, 1_000_000)
valeurs = a.tolist()

In [3]:
%timeit cumsum_python(valeurs)

79.9 ms ± 17.9 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


## La version Cython, dans le notebook

`%%cython` traduit la cellule en C, la compile et importe ses fonctions dans le notebook ; `-a` affiche en plus le code annoté : plus une ligne est jaune, plus elle fait appel à l'interpréteur Python. Les types font la différence : `double[:]` est une vue typée sur un tableau NumPy de `float64`, `cdef Py_ssize_t` déclare des entiers C, et les directives de la deuxième ligne retirent la vérification des indices.

In [4]:
%%cython -a
# cython: boundscheck=False, wraparound=False
import numpy as np


def cumsum(double[:] a):
  """Somme cumulée du tableau a, dans un nouveau tableau."""
  cdef Py_ssize_t i, n = a.shape[0]
  b = np.empty(n)
  cdef double[:] vb = b
  if n == 0:
    return b
  vb[0] = a[0]
  for i in range(1, n):
    vb[i] = vb[i - 1] + a[i]
  return b

In [5]:
print(cumsum(np.array([1.0, 2.0, 3.0, 4.0])))

[ 1.  3.  6. 10.]


Le temps de la version Cython, puis celui de `np.cumsum` :

In [6]:
%timeit cumsum(a)

3.5 ms ± 77.9 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [7]:
%timeit np.cumsum(a)

3.8 ms ± 168 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## Un module compilé avec `cythonize`

Hors d'un notebook, le même code vit dans un fichier `.pyx`, que `cythonize -i` traduit en C et compile en un module importable, rangé à côté du fichier.

In [8]:
%%writefile cumsum_cy.pyx
# cython: language_level=3, boundscheck=False, wraparound=False
import numpy as np


def cumsum(double[:] a):
  """Somme cumulée du tableau a, dans un nouveau tableau."""
  cdef Py_ssize_t i, n = a.shape[0]
  b = np.empty(n)
  cdef double[:] vb = b
  if n == 0:
    return b
  vb[0] = a[0]
  for i in range(1, n):
    vb[i] = vb[i - 1] + a[i]
  return b

Writing cumsum_cy.pyx


In [9]:
!cythonize -i cumsum_cy.pyx
import cumsum_cy

print(cumsum_cy.cumsum(np.array([1.0, 2.0, 3.0, 4.0])))



[ 1.  3.  6. 10.]


In [10]:
%timeit cumsum_cy.cumsum(a)

3.58 ms ± 103 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


Compilé dans le notebook ou en module, le code Cython est plus de dix fois plus rapide que la boucle Python, et aussi rapide que `np.cumsum`.